# ATIS classifier — train locally on your GPU (GTX 1660 Super)

Trains the YOLOv11-nano tire-safety classifier on your **local NVIDIA GPU**, using the
exact same hyperparameters as `train_model.py`, then evaluates and tunes the safety
threshold. The trained `best.pt` lands directly in the repo where `find_model_path()`
already looks — no upload/download needed.

**Run from the repo root** (`~/Desktop/ATIS`) so the relative paths resolve. In VS Code,
open this notebook and pick the project's `.venv` as the kernel (top-right "Select Kernel"),
then run the cells top to bottom. **Step 1 installs everything for you** (a CUDA build of
PyTorch + Ultralytics), so you shouldn't have to fight dependencies.

> **GTX 16-series note:** the 1660 Super (Turing, no Tensor Cores) has a known Ultralytics
> bug where mixed-precision (AMP) training silently produces `NaN` losses / 0% accuracy.
> The train cell below sets **`amp=False`** to avoid this. Leave it off on this card.

After training, run `python3 evaluate_model.py` in a terminal to merge the test metrics
into `model_card.json`.

## 1. Install dependencies (run this first)

Installs everything the notebook needs: a **CUDA** build of PyTorch (so your 1660 Super
is actually used — not CPU) plus Ultralytics, which itself pulls numpy, opencv, pillow,
pyyaml, matplotlib, pandas, etc. It **auto-detects a CPU-only torch and replaces it**, and
no-ops when everything is already present, so it's safe to re-run.

> Uses CUDA 12.1 wheels (`cu121`), which the 1660 Super supports. If your NVIDIA driver is
> older and the install errors out, change `CUDA = "cu121"` to `"cu118"` in the cell below.
>
> If this cell reports it **updated torch**, use the toolbar to **Restart Kernel**, then run
> from the top — Python can't hot-swap a library that's already loaded.

In [ ]:
# --- Self-contained dependency install (run once) ---
# Forces a CUDA build of torch/torchvision so the GPU is used, then Ultralytics.
# Detects torch by its distribution metadata WITHOUT importing it, so a clean
# top-to-bottom run needs no kernel restart.
import importlib.metadata as _md
import importlib.util as _iu
import subprocess, sys

CUDA = "cu121"   # 1660 Super supports this; switch to "cu118" for older drivers

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

did_install = False

# torch + torchvision: install if missing OR if a CPU-only wheel ("+cpu") is present
torch_v = _ver("torch")
if torch_v is None or "+cpu" in torch_v:
    print(f"torch = {torch_v!r}  ->  installing CUDA build ({CUDA}) ...")
    _pip("--index-url", f"https://download.pytorch.org/whl/{CUDA}", "torch", "torchvision")
    did_install = True
else:
    print(f"torch OK: {torch_v}")

# ultralytics (brings numpy, opencv-python, pillow, pyyaml, matplotlib, pandas, scipy, ...)
if _iu.find_spec("ultralytics") is None:
    print("installing ultralytics ...")
    _pip("ultralytics")
    did_install = True
else:
    print(f"ultralytics OK: {_ver('ultralytics')}")

if did_install and "torch" in sys.modules:
    print("\n***  Packages updated, but torch is already loaded in this kernel.")
    print("***  RESTART THE KERNEL (toolbar), then run from the top.       ***")
elif did_install:
    print("\nDone installing. Continue to the next cell.")
else:
    print("\nAll dependencies already satisfied.")

## 2. Verify GPU

Confirms PyTorch can see the card. You want `CUDA OK -> NVIDIA GeForce GTX 1660 SUPER`.
If it still says CPU-only right after the install cell, **Restart Kernel** and run again.

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"CUDA OK -> {name}  ({vram:.1f} GB VRAM)")
    print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
else:
    print("CUDA NOT available -- torch would run on CPU (very slow).")
    print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
    print(">> Re-run the install cell above, then Restart Kernel and run from the top.")

# Driver-level sanity check (works on Windows/Linux with NVIDIA drivers installed):
!nvidia-smi -L

## 3. Locate the local dataset

No upload needed — the dataset is already on disk. This cell finds `ATIS_Dataset/`
relative to the repo and prints per-split / per-class counts as a sanity check.

If you haven't built the dataset yet, run `python3 prepare_dataset.py` in a terminal first.

In [ ]:
import os
from pathlib import Path

# Resolve the repo root whether the kernel's cwd is the repo root or the notebook dir.
candidates = [Path.cwd() / "ATIS_Dataset", Path.cwd().parent / "ATIS_Dataset"]
DATA_DIR = next((str(c) for c in candidates if c.is_dir()), str(candidates[0]))
assert os.path.isdir(DATA_DIR), (
    f"ATIS_Dataset not found near {Path.cwd()}.\n"
    "Run this notebook from the repo root, or run: python3 prepare_dataset.py"
)
print("Dataset:", DATA_DIR)

# sanity: per-split per-class counts
for split in ('train', 'val', 'test'):
    for cls in ('normal', 'cracked'):
        d = os.path.join(DATA_DIR, split, cls)
        n = len([f for f in os.listdir(d) if not f.startswith('.')]) if os.path.isdir(d) else 0
        print(f"{split}/{cls}: {n}")

## 4. Train

Identical hyperparameters to `train_model.py`: 100 epochs (early stop patience 20),
imgsz 224, batch 32, seed 0, deterministic, cosine LR, 20° rotation aug — **plus
`amp=False`** for the GTX 16-series (see note at top).

The weights land at `ATIS_Project/tyre_safety_model/weights/best.pt` relative to the
repo, which `find_model_path()` already checks — so the app picks them up automatically.

On a 1660 Super expect roughly ~30–60s/epoch; batch 32 @ 224 fits comfortably in 6 GB.
If you hit a CUDA out-of-memory error, drop `batch` to 16. If training **hangs at start
on Windows**, set `workers=0` in the args below.

In [ ]:
import json, subprocess
from datetime import datetime, timezone
from pathlib import Path
import torch
from ultralytics import YOLO

BASE_MODEL = "yolo11n-cls.pt"
PROJECT = "ATIS_Project"
RUN_NAME = "tyre_safety_model"

DEVICE = 0 if torch.cuda.is_available() else "cpu"   # first NVIDIA GPU, else CPU

TRAIN_ARGS = {
    "data": DATA_DIR,
    "epochs": 100,
    "patience": 20,
    "imgsz": 224,
    "batch": 32,
    "seed": 0,
    "deterministic": True,
    "cos_lr": True,
    "degrees": 20.0,
    "fliplr": 0.5,
    "project": PROJECT,
    "name": RUN_NAME,
    "exist_ok": True,
    "device": DEVICE,
    "amp": False,          # GTX 16-series: AMP produces NaN losses -> keep off
    # "workers": 0,        # uncomment if training hangs at startup on Windows
}

print(f"Training on device: {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE != 'cpu' else 'CPU'})")
model = YOLO(BASE_MODEL)
results = model.train(**TRAIN_ARGS)
save_dir = Path(results.save_dir)
print("Saved to:", save_dir)

# model card (mirrors train_model.py write_model_card)
def git_sha():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], cwd=Path.cwd(), text=True
        ).strip()
    except Exception:
        return "local"

def dataset_counts():
    counts = {}
    for split in ("train", "val", "test"):
        sd = Path(DATA_DIR) / split
        if sd.is_dir():
            counts[split] = {c.name: sum(1 for _ in c.iterdir())
                             for c in sorted(sd.iterdir()) if c.is_dir()}
    return counts

card = {
    "model": "ATIS tire-safety classifier",
    "base_model": BASE_MODEL,
    "task": "classification (normal vs cracked)",
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_sha": git_sha(),
    "hyperparameters": {k: v for k, v in TRAIN_ARGS.items() if k != "data"},
    "dataset_counts": dataset_counts(),
    "weights": "weights/best.pt",
    "test_metrics": "run evaluate_model.py locally to populate",
    "trained_on": torch.cuda.get_device_name(0) if DEVICE != "cpu" else "CPU",
}
(save_dir / "model_card.json").write_text(json.dumps(card, indent=2))
print("Wrote model_card.json")

## 5. Evaluate + tune the safety threshold

Same logic as the repo's `evaluate_model.py`: per-class recall on the held-out **test**
split, then a sweep on **val** to pick the smallest `min P(normal)` cutoff that catches
>= 95% of cracked tires. The printed value is your `ATIS_CONF_THRESHOLD`.

In [ ]:
from collections import defaultdict
CLASSES = ("normal", "cracked")
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TARGET = 0.95

def gather(split):
    items = []
    for c in CLASSES:
        d = Path(DATA_DIR) / split / c
        if d.is_dir():
            for p in sorted(d.iterdir()):
                if p.suffix.lower() in EXTS:
                    items.append((p, c))
    return items

def predict_probs(m, items):
    out = []
    res = m.predict([str(p) for p, _ in items], verbose=False)
    for (_, t), r in zip(items, res):
        names = r.names; pv = r.probs.data.tolist()
        n2p = {names[i].lower(): pv[i] for i in range(len(pv))}
        out.append((t, names[int(r.probs.top1)].lower(), n2p.get("normal", 0.0)))
    return out

m = YOLO(str(save_dir / "weights" / "best.pt"))
test = predict_probs(m, gather("test"))
val = predict_probs(m, gather("val"))

# per-class recall on test (argmax)
tp = defaultdict(int); fn = defaultdict(int)
for t, p, _ in test:
    if p == t: tp[t] += 1
    else: fn[t] += 1
for c in CLASSES:
    rec = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) else 0.0
    print(f"TEST recall {c:<8} = {rec*100:5.1f}%")
print(f"--> missed-defect rate (test) = {(1 - tp['cracked']/(tp['cracked']+fn['cracked']))*100:.1f}%")

def cracked_recall(rows, thr):
    tot = sum(1 for t,_,_ in rows if t == "cracked")
    caught = sum(1 for t,_,pn in rows if t == "cracked" and pn < thr)
    return caught / tot if tot else 0.0
def normal_recall(rows, thr):
    tot = sum(1 for t,_,_ in rows if t == "normal")
    passed = sum(1 for t,_,pn in rows if t == "normal" and pn >= thr)
    return passed / tot if tot else 0.0

qualifying = [t/100 for t in range(50,100) if cracked_recall(val, t/100) >= TARGET]
chosen = min(qualifying) if qualifying else max(range(50,100), key=lambda t: cracked_recall(val, t/100))/100
print(f"\nChosen ATIS_CONF_THRESHOLD = {chosen:.2f}")
print(f"  VAL : cracked recall {cracked_recall(val,chosen)*100:.1f}% | normal recall {normal_recall(val,chosen)*100:.1f}%")
print(f"  TEST: cracked recall {cracked_recall(test,chosen)*100:.1f}% | normal recall {normal_recall(test,chosen)*100:.1f}%")

## 6. Done — weights are already in place

Because training ran locally, `best.pt` is already inside the repo at
`ATIS_Project/tyre_safety_model/weights/best.pt`, which `find_model_path()` resolves —
the app and CLI scripts will pick it up with no extra steps.

Final step (in a terminal, from the repo root):

```bash
python3 evaluate_model.py     # merges test metrics + tuned threshold into model_card.json
```

Then set `ATIS_CONF_THRESHOLD` (printed in step 5 above) in your `.env`, and run `python3 app.py`.

In [ ]:
from pathlib import Path

best = save_dir / "weights" / "best.pt"
print("Trained weights:", best.resolve())
print("Exists:", best.is_file())
print("Model card  :", (save_dir / "model_card.json").resolve())
print("\nNext: run `python3 evaluate_model.py` in a terminal to populate test metrics.")